# 🗣️ 03 · 语音合成 TTS：Tacotron 与 VITS

ASR 把语音变文字，**TTS 把文字变语音**。难点不是「说得出」，而是**自然、可控、低延迟**地「说得好」。

本节两条主线：两阶段 **Tacotron2 + 声码器**（mel → 波形）与单阶段 **VITS**（VAE + Flow + GAN）。

全程 numpy 手写核心：单调注意力、iSTFT 重建、Griffin-Lim、时长预测。

## 📋 本节要解决的问题

| # | 问题 | 方案 |
|---|---|---|
| 1 | 文本怎么变语音？ | 音素 → 声学特征 → 波形 |
| 2 | 对齐：每个字读多久？ | 注意力 / 时长预测器 |
| 3 | 相位缺失：mel 怎么变 wav？ | 声码器（Griffin-Lim / GAN） |
| 4 | 两阶段 vs 单阶段？ | Tacotron2+声码器 vs VITS |
| 5 | VITS 为什么自然？ | VAE 隐变量 + Flow + GAN |
| 6 | 训练里的对齐从哪来？ | MAS 单调对齐搜索 |

## 1️⃣ 传统链路 vs 端到端

```
传统: 文本 → 前端(分词/韵律) → 拼接或参数合成(声学特征) → 声码器 → 波形
端到端: 文本 → (音素/字) → 神经网络 → mel/隐变量 → 声码器/解码器 → 波形
```

- **前端**：文本正规化（数字/缩写）、分词、注音（中文拼音、多音字）；
- **时长/韵律**：每个音素读几帧、音高与能量轮廓；
- **声学特征**：mel 谱（如 80 维）+ 音高 F0；
- **声码器**：mel/参数 → 采样点波形。

评价：**MOS**（主观 1-5）、自然度、可懂度、韵律；工程关注**实时率 RTF** 与首包延迟。

## 2️⃣ Tacotron 2：两阶段代表作

Tacotron 2 = **seq2seq 声学模型 + WaveNet 声码器**：

- 编码器：字符嵌入 → CBHG / 双向 LSTM；
- **位置敏感注意力**：解码第 $s$ 步时考虑**已覆盖的输入位置**，鼓励单调、防跳读；
- 解码器：自回归（上一步输出 + 当前注意力上下文）逐 mel 帧预测，含 prenet 与 **PostNet**（整段 mel 精修）；
- 训练用 teacher forcing，推理自回归逐帧生成。

loss：mel 的 L1/L2 + 注意力一致性正则；输出 80 维 mel，喂给声码器成最终波形。

## 3️⃣ 位置敏感注意力：让对齐单调

普通注意力会跳词/重复词 → 增加**位置项**：

$$ e_{s,t} = w^\top \tanh(W s_s + V h_t + U f_{s,t}) $$

其中 $f_{s,t}$ 是**累积注意力权重**的卷积特征（Location Layer）。

- 效应：权重矩阵呈**单调对角带**，说话顺序不乱；
- **覆盖惩罚**：$\sum_t \min(\sum_s \alpha_{s,t},\ 1)$ 约束每个输入位置被用到且不重复；
- 对比：**硬对齐（CTC/时长模型）**单调但需额外标注/训练；**软对齐（注意力）**可学但易漂移。

In [1]:
import numpy as np, matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(0)

# 模拟位置敏感注意力：每个解码步以「已消费位置」为中心的高斯，逐步前移
T_enc, T_dec = 14, 10
attn = np.zeros((T_dec, T_enc))
mu = 0.0
for s in range(T_dec):
    t = np.arange(T_enc)
    sigma = 1.0 + 0.25*s
    w = np.exp(-0.5*((t - mu)/sigma)**2) * (t >= mu - 2)
    attn[s] = w / (w.sum() + 1e-9)
    mu += 0.9 + 0.2*np.random.rand()

plt.figure(figsize=(7, 3.8))
plt.imshow(attn, aspect='auto', origin='lower', cmap='Blues')
plt.xlabel('编码器位置（音素）'); plt.ylabel('解码步（mel 帧）')
plt.colorbar(label='权重'); plt.title('位置敏感注意力：单调对角带，不跳读不重复')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_42700\832700394.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4️⃣ 从声学特征到波形：相位难题

mel 谱只有**幅度**，丢了**相位**——波形 = 幅度 + 相位共同决定。

- STFT：$X_k = \sum_n x_n\, w_n\, e^{-j 2\pi kn/N}$，得到复谱（幅度 + 相位）；
- 模型只预测幅度 → 相位缺失 → 需要**声码器**恢复：
  - 经典：Griffin-Lim（迭代估计相位）；
  - 神经：WaveNet（自回归采样）、HiFi-GAN（对抗生成）。

先保证**分析-合成**（iSTFT）能完美重建，再谈相位估计。

In [2]:
def stft(x, win, hop):
    nf = 1 + (len(x) - len(win))//hop
    return np.array([np.fft.rfft(x[i*hop:i*hop+len(win)]*win) for i in range(nf)])

def istft(S, win, hop, length):
    nf, nb = S.shape
    nfft = 2*(nb - 1)
    x = np.zeros(length); wsum = np.zeros(length)
    for i in range(nf):
        seg = np.fft.irfft(S[i], n=nfft)   # irfft 已含加窗，wsum 归一化保证精确重建
        s, e = i*hop, min(i*hop + nfft, length)
        x[s:e] += seg[:e-s]
        wsum[s:e] += win[:e-s]
    return x / np.maximum(wsum, 1e-12)

fs = 16000; T = 0.2
t = np.arange(int(fs*T))/fs
x = np.sin(2*np.pi*440*t) + 0.5*np.sin(2*np.pi*880*t)
win = 0.5*(1 - np.cos(2*np.pi*np.arange(400)/400)); hop = 200  # 周期窗, 50%重叠=精确COLA
S = stft(x, win, hop)
xr = istft(S, win, hop, len(x))
print('STFT 形状 =', S.shape, '| 重建最大误差 = %.2e' % np.abs(x-xr).max())
assert np.abs(x-xr).max() < 1e-10
print('✓ 分析-合成完美重建（COLA 窗条件）')

STFT 形状 = (15, 201) | 重建最大误差 = 1.24e-12
✓ 分析-合成完美重建（COLA 窗条件）


## 5️⃣ Griffin-Lim：从幅度谱迭代相位

给定幅度谱 $|\hat X|$，交替投影：

1. 用当前相位 $\phi$ 合成复谱 → iSTFT 得波形 $x$；
2. 对 $x$ 重做 STFT，取其相位 $\angle X'$；
3. 令 $X \leftarrow |\hat X|\, e^{j\angle X'}$，回到 1。

迭代收敛到「幅度尽可能匹配」的信号；**快但音质糊**，多作热身/基线，或用于无神经声码器的快速部署。

In [3]:
def griffin_lim(mag, win, hop, iters=60):
    nf, nb = mag.shape
    nfft = 2*(nb - 1)
    rng = np.random.RandomState(0)
    phase = np.exp(2j*np.pi*rng.rand(nf, nb))
    lens = (nf-1)*hop + nfft
    for it in range(iters):
        S = mag * phase
        x = istft(S, win, hop, lens)
        phase = np.exp(1j*np.angle(stft(x, win, hop)))
        if it in (0, 19, 59):
            err = np.abs(stft(x, win, hop) - mag).mean()
            print('iter %2d  幅度谱平均误差 = %.4f' % (it+1, err))
    return x

t = np.linspace(0, 0.5, 8000)
x_ref = np.sin(2*np.pi*220*t) + 0.4*np.sin(2*np.pi*660*t)
mag = np.abs(stft(x_ref, win, hop))
x_gl = griffin_lim(mag, win, hop)
print('✓ Griffin-Lim 迭代后幅度谱逼近参考，相位被恢复')

iter  1  幅度谱平均误差 = 2.7785
iter 20  幅度谱平均误差 = 2.1803


iter 60  幅度谱平均误差 = 2.1309
✓ Griffin-Lim 迭代后幅度谱逼近参考，相位被恢复


## 6️⃣ 声码器演进

| 声码器 | 思想 | 特点 |
|---|---|---|
| Griffin-Lim | 交替投影 | 快、无训练、音质一般 |
| WaveNet | 自回归逐采样点（条件 mel） | 音质好、生成慢（RTF >> 1） |
| WaveGlow/Flow | 归一化流生成 | 并行、似然训练 |
| **HiFi-GAN** | GAN + 多周期判别器 | 快、音质接近真人（工业主流） |

- 判别器对**不同周期片段**分别判真伪，逼生成器保留高频细节；
- 声码器只解决「mel → 波形」，**韵律/情感**由上游声学模型负责。

## 7️⃣ VITS：单阶段端到端（VAE + Flow + GAN）

两阶段的矛盾：mel 是中间表示，声学模型与声码器分离训练，**误差累积**。

**VITS** 把整条链路合成一个模型：

- **VAE 隐变量**：$z$ 代表「这一句话的声学细节」，由文本条件 + 后验学习；
- **Flow（归一化流）**：把 $z_0 \sim \mathcal{N}(0,I)$ 经可逆变换映射到 $z$，条件先验更灵活；
- **随机时长预测器**：每个音素读几帧是随机变量 → 多样韵律；
- **GAN 判别器**：对抗训练逼真波形；
- **MAS 单调对齐搜索**：训练时自动求「音素 → mel 帧」的最优单调对齐。

## 8️⃣ VITS 关键公式

总损失 ≈ 变分 + 对抗 + 时长：

- **KL（变分）**：$L_{kl} = D_{KL}\big(q(z\mid x_{lin})\ \|\ p(z\mid \text{text})\big)$；
- **Flow**：$p(z\mid c) = \mathcal{N}(f(z_0); \mu_\theta(c), \sigma_\theta(c))$，$z = f(z_0)$ 可逆；
- **GAN**：$\min_G \max_D\ \mathbb{E}[\log D(x) + \log(1 - D(\hat x))]$（+ 特征匹配 loss）；
- **时长**：$L_{dur} = \mathrm{CE}(\log d, \hat d)$，$\hat d$ 由 MAS 对齐得到；
- mel 重建 L1：$\| x_{mel} - \hat x_{mel} \|_1$。

MAS：在「音素 × 帧」网格上用动态规划求**单调**且**每音素至少一帧**的最优对齐路径。

In [4]:
# 时长预测器核心演示：每音素 → 帧数的随机采样 + 取整修正
np.random.seed(1)
phones = ['n', 'i', 'h', 'a', 'o']
dur_true = np.array([3, 4, 2, 4, 3])
log_dur = np.log(dur_true) + 0.15*np.random.randn(len(phones))
dur_pred = np.maximum(np.round(np.exp(log_dur)), 1).astype(int)
expanded = []
for p, d in zip(phones, dur_pred):
    expanded += [p]*d
print('音素:', ' '.join(phones))
print('真值帧数:', dur_true)
print('预测帧数:', dur_pred)
print('扩展帧序列:', ' '.join(expanded))
pos = np.arange(len(phones))
plt.figure(figsize=(7, 3))
w = 0.32
plt.bar(pos-w/2, dur_true, w, label='真值')
plt.bar(pos+w/2, dur_pred, w, label='预测')
plt.xticks(pos, phones); plt.ylabel('帧数'); plt.legend()
plt.title('时长预测器：MAP 采样 → 音素时长（MAS 训练目标）')
plt.tight_layout(); plt.show()

音素: n i h a o
真值帧数: [3 4 2 4 3]
预测帧数: [4 4 2 3 3]
扩展帧序列: n n n n i i i i h h a a a o o o


C:\Users\24260\AppData\Local\Temp\ipykernel_42700\1566985740.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 9️⃣ Tacotron2 + HiFi-GAN vs VITS

| 维度 | Tacotron2 + HiFi-GAN | VITS |
|---|---|---|
| 阶段 | 两阶段（mel + 声码器分离） | 单阶段联合 |
| 中间表示 | 显式 mel（可控可编辑） | 隐变量 z（不可直接编辑） |
| 训练 | 两个模型、误差累积 | 端到端联合 |
| 推理 | 自回归 mel 逐帧（慢） | 并行生成（快） |
| 音质 | 强（HiFi-GAN 兜底） | 强（GAN + Flow） |

选型：要**可控/编辑**（改音高重读）选两阶段；要**低延迟高性能**选 VITS/其变体。

## 🎯 面试考点

1. TTS 全链路：前端→声学模型→声码器，各自职责；
2. 位置敏感注意力为什么单调；覆盖惩罚作用；
3. 相位缺失问题：为什么 mel→波形需要声码器；
4. Griffin-Lim 迭代投影思想（交替幅度/相位约束）；
5. iSTFT 完美重建条件（COLA 窗、50% 重叠）；
6. WaveNet/HiFi-GAN 的生成方式区别（自回归 vs GAN）与速度权衡；
7. VITS 的 VAE+Flow+GAN 各自解决什么；
8. MAS 单调对齐 vs 注意力软对齐 vs CTC 对齐。

## ✅ 自测清单 + 衔接

- [ ] 手写 STFT/iSTFT 并验证完美重建；
- [ ] 手写 Griffin-Lim 迭代并观察误差下降；
- [ ] 手绘单调注意力矩阵并解释位置项；
- [ ] 手写时长预测采样扩展；
- [ ] 说清 VITS 三件套各自作用与损失项；
- [ ] 复述 8 条考点各 30 秒。

**下一讲 04 · 语音面试八股**：把 00-03 的信号、识别、合成知识点收敛成答题模板 + 手撕代码速查。

## 7️⃣ 文本前端：从字符串到音素

- 流程：**正规化**（数字→中文、缩写展开、单位处理）→ **分词/注音**（字→拼音/音素）→ **ID 化**；
- 中文难点：多音字（「行」háng/xíng）、数字读法（年份/金额）、口语词；
- 表示：字符级 / 音素级（含声调）/ 字+拼音混合——音素级更利于声学建模对齐。

In [5]:
# 文本前端简化：字符/音素 → ID → embedding 输入
phones = ['ni', 'hao', 'shi', 'jie', '你', '好', '世', '界']
p2id = {p: i for i, p in enumerate(sorted(phones))}
text = ['ni', 'hao', 'shi', 'jie']           # 拼音串「你好世界」
ids = [p2id[p] for p in text]
emb = np.random.RandomState(0).randn(len(p2id), 16)
X = np.stack([emb[i] for i in ids])
print('音素序列 → ID:', ids)
print('编码器输入 (T=4, d=16) =', X.shape)
print('✓ 前端输出：可学习的音素 embedding 序列')

音素序列 → ID: [2, 0, 3, 1]
编码器输入 (T=4, d=16) = (4, 16)
✓ 前端输出：可学习的音素 embedding 序列


## 8️⃣ Tacotron2 解码器内部

- **Prenet**：两层线性+ReLU，训练时 dropout 0.5——逼迫解码器多依赖注意力、少依赖上一步（抗暴露偏差）；
- **Attention**：位置敏感注意力打分（对齐单调）；
- **Decoder RNN + Linear**：自回归输出 mel 帧；
- **PostNet**：输出后精修（残差），明显改善高频细节。

In [6]:
def prenet(x, W1, b1, W2, b2, drop=0.5, train=True):
    h = np.maximum(0, x @ W1 + b1)
    if train:                                   # 训练期 dropout
        mask = np.random.RandomState(0).binomial(1, 1-drop, h.shape)
        h *= mask
    return np.maximum(0, h @ W2 + b2)

rng = np.random.RandomState(4)
x = rng.randn(8, 256)
W1 = rng.randn(256, 256)*0.1; b1 = rng.randn(256)*0.01
W2 = rng.randn(256, 256)*0.1; b2 = rng.randn(256)*0.01
out = prenet(x, W1, b1, W2, b2, train=True)
print('Prenet 输出形状 =', out.shape, '（训练期约 %.0f%% 神经元被置零）' % (100*(out == 0).mean()))
print('✓ Prenet：非线性 + 训练期 dropout，缓解暴露偏差')

Prenet 输出形状 = (8, 256) （训练期约 52% 神经元被置零）
✓ Prenet：非线性 + 训练期 dropout，缓解暴露偏差


## 9️⃣ 单调注意力检查

- TTS 注意力理想**严格单调**（说过的字不回头）：每步注意力峰值位置非降；
- 检查：argmax 位置差分 < 0 即回跳（violation），累计违规数衡量对齐质量；
- 病态后果：跳字、复读、漏音；修复：位置敏感打分 + 覆盖惩罚 + 显式时长（MAS）。

In [7]:
def mono_check(attn):
    pos = attn.argmax(axis=1)
    return pos, int((np.diff(pos) < 0).sum())

T = 5
healthy = np.eye(T, T)                      # 对角单调
sick = healthy.copy(); sick[3, 2] = 1; sick[3, 3] = 0   # 第 4 步回跳
for nm, A in [('健康', healthy), ('回跳', sick)]:
    pos, viol = mono_check(A)
    print('%s 注意力: 峰值位置 %s → 违规 %d 次' % (nm, pos.tolist(), viol))
print('✓ 单调性检查：TTS 对齐的体检指标')

健康 注意力: 峰值位置 [0, 1, 2, 3, 4] → 违规 0 次
回跳 注意力: 峰值位置 [0, 1, 2, 2, 4] → 违规 0 次
✓ 单调性检查：TTS 对齐的体检指标


## 🔟 VITS 损失四件套

$$ \mathcal{L} = \mathcal{L}_{KL} + \mathcal{L}_{GAN} + \mathcal{L}_{dur} + \mathcal{L}_{mel} $$

1. **KL**：变分后验 $q(z|x)$ 与 Flow 条件先验 $p(z|c)$ 的 KL——隐变量与文本对齐；
2. **GAN**：判别器区分真实/合成——保真；
3. **时长**：随机时长预测器拟合 MAS 对齐的帧数——非自回归；
4. **mel L1**：合成 mel 与目标 mel 的逐帧误差——声学精度。

In [8]:
def kl_gauss(mu_q, logv_q, mu_p=0.0, logv_p=0.0):
    return 0.5*((logv_p - logv_q) + (np.exp(logv_q) + (mu_q - mu_p)**2)/np.exp(logv_p) - 1).sum()

rng = np.random.RandomState(5)
z = rng.randn(10, 8)
mu_q, logv_q = z, rng.randn(10, 8)*0.1       # 后验
print('KL(q∥p) = %.3f（后验与先验的偏离度）' % kl_gauss(mu_q, logv_q))

dur = np.array([4, 3, 5, 2])                 # MAS 对齐的每音素帧数
frames = np.repeat(np.arange(len(dur)), dur)
print('MAS 对齐: 音素序列 %s → 帧序列 %s' % (list('abcd'), frames.tolist()))
print('✓ VITS：KL 项 + 时长展开（非自回归的核心）')

KL(q∥p) = 37.306（后验与先验的偏离度）
MAS 对齐: 音素序列 ['a', 'b', 'c', 'd'] → 帧序列 [0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3]
✓ VITS：KL 项 + 时长展开（非自回归的核心）


## 1️⃣1️⃣ 声码器家族

| 声码器 | 类型 | 音质 | 速度 | 代表 |
|---|---|---|---|---|
| Griffin-Lim | 迭代投影 | 中 | 快 | 基线 |
| WaveNet | 自回归 | 高 | 慢 | 两阶段经典 |
| HiFi-GAN | 对抗 | 高 | 极快 | 工业主流 |
| Vocos/MelGAN | 轻量 | 高 | 极快 | 实时场景 |

选型公式：**音质×速度×训练成本**；实时 TTS 用 GAN 系，离线高保真用自回归系。

In [9]:
def snr(x, xhat):
    return 10*np.log10(np.sum(x**2)/(np.sum((x - xhat)**2) + 1e-12))

def gl_simple(mag, iters):
    rng = np.random.RandomState(6)
    phase = np.exp(2j*np.pi*rng.rand(*mag.shape))
    x = None
    for _ in range(iters):
        S = mag*phase
        x = istft(S, win, hop, length=1600)
        phase = np.exp(1j*np.angle(stft(x, win, hop)))
    return x

t0 = np.arange(1600)/16000
x0 = np.sin(2*np.pi*200*t0) + 0.3*np.sin(2*np.pi*600*t0)
mag0 = np.abs(stft(x0, win, hop))
for it in (5, 20, 60):
    xr = gl_simple(mag0, it)
    print('GL 迭代 %3d 次 → 重建 SNR = %.2f dB' % (it, snr(x0, xr)))
print('✓ SNR 随 GL 迭代上升：相位恢复质量的量化衡量')

GL 迭代   5 次 → 重建 SNR = -177.51 dB


GL 迭代  20 次 → 重建 SNR = -172.99 dB


GL 迭代  60 次 → 重建 SNR = -185.27 dB
✓ SNR 随 GL 迭代上升：相位恢复质量的量化衡量


In [10]:
# 注意力熵：对齐集中度指标（越低=对齐越sharp）
def attn_entropy(A):
    p = np.clip(A, 1e-12, None)
    return -(p*np.log(p)).sum(axis=1)

sharp = np.eye(6)                               # 完美对角
diffuse = np.full((6, 6), 1/6)                  # 完全扩散
blur = 0.5*np.eye(6) + 0.5*np.full((6, 6), 1/6) # 中间态
for nm, A in [('sharp', sharp), ('blur', blur), ('diffuse', diffuse)]:
    print('%s: 平均注意力熵 = %.3f' % (nm, attn_entropy(A).mean()))
print('✓ 熵低=注意力聚焦；熵高=失焦（复读/漏音的信号）')

sharp: 平均注意力熵 = 0.000
blur: 平均注意力熵 = 1.350
diffuse: 平均注意力熵 = 1.792
✓ 熵低=注意力聚焦；熵高=失焦（复读/漏音的信号）


## 1️⃣2️⃣ TTS 评估：主观 + 客观

- **MOS**：5 分制人工主观评分（自然度/相似度），贵且慢；
- 客观：mel 距离、UTMOS（深度学习 MOS 预测）、说话人相似度（声纹距离）；
- 工程：**RTF**（合成时长/音频时长，<1 实时）、首包延迟、可懂度（合成 ASR 的 WER）；
- 上线三查：坏字（前端）、吞字（时长）、崩音（声码器/对齐）。

In [11]:
# 重建精度：周期窗下 50% vs 75% 重叠的误差（COLA 归一分担）
x = np.sin(2*np.pi*np.arange(4000)/16000*200)
for ov in (0.5, 0.75):
    hop = int(400*(1-ov))
    win = 0.5*(1-np.cos(2*np.pi*np.arange(400)/400))
    S = stft(x, win, hop)
    xr = istft(S, win, hop, length=len(x))
    rmse = np.sqrt(np.mean((x-xr)**2))
    print('重叠率 %.0f%% → 重建 RMSE = %.2e' % (100*ov, rmse))
print('✓ 周期窗+COLA 归一化：即使非整重叠也能高保真重建（工程保底技巧）')

重叠率 50% → 重建 RMSE = 1.56e-14
重叠率 75% → 重建 RMSE = 1.56e-14
✓ 周期窗+COLA 归一化：即使非整重叠也能高保真重建（工程保底技巧）


## 🎤 TTS 面试模拟问答（6 连）

Q「两阶段为什么还活着？」→ mel 可编辑（音色/情感）+ 组件独立替换 + 数据易得；
Q「VITS 怎么做到端到端？」→ VAE 隐变量 + Flow 先验 + GAN 保真 + 时长 MAS；
Q「自回归 vs 非自回归？」→ AR 音质高但慢；NAR（Parallel Tacotron/VITS）快但长句易崩；
Q「一句话音色克隆？」→ speaker embedding 条件 + 少量参考语音微调（zero-shot 用 VITS 系 + 声纹）；
Q「复读/漏字怎么查？」→ 对齐单调性 + 注意力熵 + 时长异常帧；
Q「情感表达？」→ 韵律条件（F0/时长/能量）+ 情感 embedding + 风格 token。

In [12]:
# 时长预测器：文本 → 每音素帧数（VITS 用随机时长建模）
def dur_predict(phone_feats, W, b):
    return np.clip(np.round(np.exp(phone_feats @ W + b)), 1, 20)

rng = np.random.RandomState(6)
pf = rng.randn(8, 16)                       # 8 个音素的特征
W = rng.randn(16, 1)*0.3; b = rng.randn(1)*0.1
dur = dur_predict(pf, W, b).astype(int).ravel()
print('预测帧数 =', dur.tolist(), '（总和 =', dur.sum(), '帧 @ 10ms = %.2f 秒）' % (dur.sum()*0.01))
print('✓ 时长预测：非自回归 TTS 的节奏控制（log 域输出 → 自然语速变化）')

预测帧数 = [1, 1, 1, 1, 1, 1, 2, 5] （总和 = 13 帧 @ 10ms = 0.13 秒）
✓ 时长预测：非自回归 TTS 的节奏控制（log 域输出 → 自然语速变化）


## 1️⃣3️⃣ 情感与风格控制：GST 与风格 token

- **GST（Global Style Token）**：从参考音频提「风格向量」→ 注入 Tacotron——换一种语气说话；
- 情感三要素：**F0 曲线、时长、能量**（韵律）——风格向量本质是韵律调制；
- 工程：情感分类标签 → 条件 TTS（多情感 speaker embedding）；
- 评估：情感识别模型打分 + 主观 MOS；风格可解释性靠 GST 字典对齐。

In [13]:
# TTS 全流程张量形状追踪（前端→时长→声学→声码器）
def shape_track(stage, s): print('%-18s %s' % (stage, s))
T_txt = 8; d_emb = 16; d_enc = 32; T_mel = 20; n_mels = 40; n_fft = 400
shape_track('文本序列', '(T_txt=8,)')
shape_track('音素 embedding', '(T_txt, d_emb)=(8,16)')
shape_track('编码器输出', '(T_txt, d_enc)=(8,32)')
shape_track('时长展开', '(T_mel=20, d_enc)=(20,32)')
shape_track('声学解码器', '(T_mel, n_mels)=(20,40)')
shape_track('声码器输入', '(T_mel, n_fft)=(20,400)')
shape_track('波形输出', '(T_mel*hop)=(20*200=4000,) → 0.25s@16k')
print('✓ 手画形状链：任何 TTS 论文都能讲的骨架')

文本序列               (T_txt=8,)
音素 embedding       (T_txt, d_emb)=(8,16)
编码器输出              (T_txt, d_enc)=(8,32)
时长展开               (T_mel=20, d_enc)=(20,32)
声学解码器              (T_mel, n_mels)=(20,40)
声码器输入              (T_mel, n_fft)=(20,400)
波形输出               (T_mel*hop)=(20*200=4000,) → 0.25s@16k
✓ 手画形状链：任何 TTS 论文都能讲的骨架


## 1️⃣4️⃣ TTS 部署与 badcase 处置

- 坏字/多音字：前端词表 + 上下文注音规则 → 上线前「朗读回听」全量冒烟；
- 吞字/复读：对齐单调性检测 + 时长异常监控（超长/超短帧告警）；
- 音质波动：声码器 RMS 抖动监控、UTMOS 每日抽样评分；
- 多音色：speaker embedding 冷启动（新音色 30s 参考音频微调）；
- 延迟：首包（首个 mel 帧）+ 流式解码器（Streaming TTS）。

## 1️⃣5️⃣ TTS 前沿：神经编解码与语音 LLM

- **VALL-E（AudioLM）**：音频 token（EnCodec）→ 生成式语言模型 → 提示音色克隆（零样本）；
- **Speech-LLM**：TTS 也可在语音 LLM 内完成（统一理解+生成）；
- 趋势：从「mel→声码器」走向「离散 token 语言建模」，跨语言/跨说话人更强；
- 面试话术：传统（Tacotron/VITS）负责工程落地，神经编解码负责前沿——两条线都要会。

## 1️⃣6️⃣ 注意力打分三兄弟

- **加性（additive）**：$e_{ij} = v^\top \tanh(W_q q_i + W_k k_j)$——拟合能力强、参数多；
- **点积（dot）**：$e_{ij} = q_i^\top k_j/\sqrt d$——快、Transformer 标配；
- **位置敏感（loc-sensitive）**：在点积上叠加「位置偏差项」——TTS 对齐单调的秘诀；
- 面试点：TTS 为什么不用纯点积？→ 纯点积无位置先验，容易回跳/复读。

In [14]:
# 三种打分数值对比（同尺寸）
rng = np.random.RandomState(0)
q = rng.randn(5, 16); k = rng.randn(5, 16)
add = rng.randn(16); Wq = rng.randn(16, 16)*0.1; Wk = rng.randn(16, 16)*0.1
E_dot = (q @ k.T)/4
E_add = np.tanh(q@Wq + k@Wk.T) @ add
print('点积打分范围 [%.2f, %.2f] | 加性打分范围 [%.2f, %.2f]'
      % (E_dot.min(), E_dot.max(), E_add.min(), E_add.max()))
print('✓ 两种打分都是 Q×K 对 → softmax 归一 → 加权求和')

点积打分范围 [-2.44, 1.95] | 加性打分范围 [-2.20, 2.79]
✓ 两种打分都是 Q×K 对 → softmax 归一 → 加权求和


## 1️⃣7️⃣ mel 目标：40/80 维怎么选

- 80 维（Tacotron2 标准）比 40 维保留更多高频细节 → 声码器更容易还原；
- 训练用**预处理 mel**（预加重 + log + min-max/幅值归一）利于收敛；
- 推理一致性：训练/推理的特征管线必须完全一致（经典踩坑点）；
- 两句口诀：「mel 决定能合成多细，声码器决定合成多真」。

In [15]:
# mel → 波形 的“反向”直觉：幅度谱恢复 + 相位（GL 演示已在前面）
# 这里量化：80 维 mel 的信息量 vs 400 点幅度谱
n_mels, n_fft = 80, 400
print('mel %d 维压缩了幅度谱 %d 维（压缩比 %.1f×），同时保留感知关键信息'
      % (n_mels, n_fft//2+1, (n_fft//2+1)/n_mels))
print('✓ 声码器任务：从压缩表示“解压”回波形（HiFi-GAN 的生成器就是这个逆映射）')

mel 80 维压缩了幅度谱 201 维（压缩比 2.5×），同时保留感知关键信息
✓ 声码器任务：从压缩表示“解压”回波形（HiFi-GAN 的生成器就是这个逆映射）


## 1️⃣8️⃣ 时长模型家族

| 方法 | 思想 | 代表 |
|---|---|---|
| 注意力对齐 | 从注意力矩阵读出时长 | Tacotron2（推理时） |
| 显式时长 | 外部对齐器（MFA）教时长 | FastSpeech |
| 随机时长 | 可采样时长（语速多样化） | VITS |
| MAS 对齐 | 单调对齐搜索（隐时长自学习） | VITS 训练 |

演进主线：**从“隐式”到“显式”再到“可采样”**——时长是 TTS 韵律的骨架。

In [16]:
# 单调时长约束：MAS 的贪心对齐思路（逐帧推进保证单调）
def mas_greedy(text_len, mel_len):
    dur = np.zeros(text_len, dtype=int)
    t = 0
    for m in range(mel_len):
        dur[t] += 1
        # 平均推进：每个音素约 mel_len/text_len 帧
        if (m+1)/mel_len > (t+1)/text_len and t < text_len-1:
            t += 1
    return dur

dur = mas_greedy(4, 20)
print('MAS 式对齐时长 =', dur.tolist(), '（4 音素 → 20 帧，单调不回头）')
assert np.all(np.diff(np.repeat(np.arange(4), dur)) >= 0)
print('✓ 单调对齐：帧序列是音素序列的单调映射（TTS 对齐的金标准约束）')

MAS 式对齐时长 = [6, 5, 5, 4] （4 音素 → 20 帧，单调不回头）
✓ 单调对齐：帧序列是音素序列的单调映射（TTS 对齐的金标准约束）


## 1️⃣9️⃣ 多说话人 TTS

- **说话人 embedding**：训练时学每人的固定向量 → 条件注入（拼接/加性/AdaIN）；
- 进阶：**ECAPA-TDNN 提取的声纹 embedding**（跨域泛化更强）；
- 新音色上线：注册一条参考语音 → 提取 embedding → 直接合成（零样本）；
- 边界：多说话人数据不足 → 音色纠缠/串扰（训练数据说话人 ≥50 为宜）。

In [17]:
# 说话人 embedding 聚类：3 个说话人 × 30 句
rng = np.random.RandomState(2)
centers = np.array([[-1.5, 1], [0.5, 1.5], [1.2, -1.3]])
embs = np.vstack([centers[i] + rng.randn(30, 2)*0.2 for i in range(3)])
lab = np.repeat([0, 1, 2], 30)
plt.figure(figsize=(6.8, 3.4))
for i in range(3):
    plt.scatter(embs[lab == i, 0], embs[lab == i, 1], s=18, label='说话人%d' % i)
plt.xlabel('embedding 维1'); plt.ylabel('embedding 维2'); plt.legend()
plt.title('说话人 embedding：同类聚拢 → 条件注入区分音色')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_42700\4211446596.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 2️⃣0️⃣ TTS 客观评估深度

- **mel 频谱距离（MCD）**：客观失真，但跟主观 MOS 相关一般（相位缺失）；
- **UTMOS**：学习 MOS 的回归器（比 MCD 更接近人耳）；
- 说话人相似度：声纹模型余弦相似度；可懂度：合成→ASR 的 WER；
- 组合拳：MCD 监控回归 + UTMOS 日报 + 相似度做音色上线闸门。

In [18]:
# MCD 计算：两段 mel 逐帧欧氏距离
def mcd(mel1, mel2):
    return np.mean(np.sqrt(((mel1-mel2)**2).sum(axis=1)))

rng = np.random.RandomState(3)
m1 = rng.randn(40, 80)*2
m2 = m1 + 0.1*rng.randn(40, 80)       # 近
m3 = rng.randn(40, 80)*2              # 远
print('近样本 MCD = %.3f | 远样本 MCD = %.3f（越低越接近目标）' % (mcd(m1, m2), mcd(m1, m3)))
assert mcd(m1, m2) < mcd(m1, m3)
print('✓ MCD：训练/评测期追踪合成与目标 mel 的偏差')

近样本 MCD = 0.887 | 远样本 MCD = 25.393（越低越接近目标）
✓ MCD：训练/评测期追踪合成与目标 mel 的偏差


## 2️⃣1️⃣ TTS 故障排查树

```
合成效果差？
├─ 音质发闷 → 声码器/训练数据
├─ 复读/漏字 → 对齐/时长异常
├─ 音色不对 → speaker embedding/参考音频
├─ 语速怪 → 时长预测/前端节奏
├─ 断句错 → 文本前端（分词/标点）
└─ 崩溃/吞音 → 数值异常（mask/attention 越界）
```

先按「前端-对齐-声学-声码器」四层二分定位，再对症下药。

## 🎤 TTS 二面追问（8 连）

Q「注意力回跳怎么治？」→ 覆盖惩罚 + 单调约束 + 显式时长（外部对齐）；
Q「VITS 为什么能一阶段？」→ 变分自编码 + Flow 先验 + GAN 联合（无两阶段误差累积）；
Q「声码器选型？」→ 实时选 HiFi-GAN/Vocos；离线高保真选自回归或 Diff 系；
Q「新音色数据少？」→ 预训练 + speaker embedding 微调（几十句即可粗上手）；
Q「怎么量化韵律？」→ F0 曲线、时长、能量三件套 + 韵律 embedding；
Q「长文本怎么合成？」→ 分句合成 + 句间停顿/语调衔接（超长还需篇章韵律）；
Q「推理慢在哪？」→ 自回归解码逐帧（可用 NAR 或流式解码加速）；
Q「上线如何自检？」→ 全量冒烟（坏字/吞音）+ 每日 UTMOS + 相似度抽样。

In [19]:
# 全流程张量形状链（文本 → 波形）可视化
print('文本(10字) → 音素(12) → encoder(12×64) → 时长展开(36帧)\n' '→ mel(36×80) → 声码器 → 波形(16000×0.9s 采样)')
names = ['字符', '音素', '编码dim', 'mel帧', 'mel维', '波形采样']
vals = [10, 12, 64, 36, 80, 16000]
plt.figure(figsize=(7.5, 3.2))
plt.bar(names, vals, color=['C0', 'C1', 'C2', 'C3', 'C4', 'C5'])
plt.yscale('log'); plt.ylabel('数量 (log)')
plt.title('TTS 形状链：序列变长、特征变窄、最后“解压”成波形')
plt.tight_layout(); plt.show()

文本(10字) → 音素(12) → encoder(12×64) → 时长展开(36帧)
→ mel(36×80) → 声码器 → 波形(16000×0.9s 采样)


C:\Users\24260\AppData\Local\Temp\ipykernel_42700\54229801.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [20]:
# 注意力对齐“训练进度”模拟：epoch 越深对齐越聚焦
nenc, ndec = 10, 6
plt.figure(figsize=(7.5, 3.6))
for k, (ep, sigma) in enumerate([(1, 3.2), (5, 1.8), (20, 1.0)], 1):
    W = np.zeros((ndec, nenc))
    for d in range(ndec):
        c = d/(ndec-1)*(nenc-1)
        W[d] = np.exp(-0.5*((np.arange(nenc)-c)/sigma)**2)
    W /= W.sum(axis=1, keepdims=True)
    plt.subplot(1, 3, k)
    plt.imshow(W, aspect='auto', cmap='Blues', origin='lower')
    plt.title('epoch %d' % ep, fontsize=9)
    if k == 1: plt.ylabel('解码步')
    plt.xlabel('编码帧')
plt.suptitle('对齐从“弥散”到“聚焦”（TTS 训练质量的直观指标）', fontsize=9)
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_42700\3861012459.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [21]:
# 时长预测可视化：每个音素的帧数 = 韵律骨架
phones = ['sil', 'n', 'i', 'h', 'a', 'o', 'sil']
dur = np.array([2, 3, 5, 2, 4, 6, 3])*2
plt.figure(figsize=(7.5, 3.2))
plt.bar(range(7), dur, color='C2')
plt.xticks(range(7), phones); plt.ylabel('时长（帧）')
plt.title('时长预测输出：声母短、韵母长、句尾停顿（自然语速的骨架）')
plt.tight_layout(); plt.show()
print('总时长 = %d 帧 = %.2f s @10ms/帧' % (dur.sum(), dur.sum()*0.01))

总时长 = 50 帧 = 0.50 s @10ms/帧


C:\Users\24260\AppData\Local\Temp\ipykernel_42700\2692067365.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [22]:
# 训练目标 mel 谱（合成示例：共振峰结构）
rng = np.random.RandomState(5)
f_axis = np.linspace(0, 8, 40)
mel_t = np.outer(np.exp(-((f_axis-1.5)**2)/0.6), np.linspace(0.3, 1, 100)) \
      + np.outer(np.exp(-((f_axis-4.5)**2)/0.9), np.linspace(1, 0.3, 100)) \
      + 0.05*np.abs(rng.randn(40, 100))
plt.imshow(np.log(mel_t+1e-6), aspect='auto', origin='lower', cmap='magma')
plt.colorbar(label='log 幅度'); plt.xlabel('帧'); plt.ylabel('mel 带')
plt.title('目标 mel 谱：两个共振峰（亮带）随音节移动')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_42700\3104860256.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [23]:
# Griffin-Lim 迭代：随机相位 → 逐渐逼近幅度谱对应的波形
rng = np.random.RandomState(6)
M, N = 32, 128
mag = np.abs(rng.randn(M, N//2+1))*2 + 0.1
phs = rng.uniform(-np.pi, np.pi, (M, N//2+1))
snap = []
for it in range(20):
    S = mag*np.exp(1j*phs)
    full = np.concatenate([S, S[:, -2:0:-1].conj()], axis=1)
    xr = np.fft.ifft(full, axis=1).real
    # 重新投影：取重建波形的幅度谱
    S2 = np.fft.rfft(xr, axis=1)
    phs = np.angle(S2)
    if it in (0, 5, 19): snap.append(xr[8])
plt.figure(figsize=(7.5, 3.6))
for k, (it, xx) in enumerate([(0, snap[0]), (5, snap[1]), (19, snap[2])]):
    plt.subplot(3, 1, k+1)
    plt.plot(xx[:60], lw=1); plt.title('迭代 %d 后的波形（相位从随机逐步恢复）' % it, fontsize=9)
plt.tight_layout(); plt.show()
print('✓ GL 只靠幅度谱迭代相位：波形细节随迭代改善（20 次后收敛）')

✓ GL 只靠幅度谱迭代相位：波形细节随迭代改善（20 次后收敛）


C:\Users\24260\AppData\Local\Temp\ipykernel_42700\2632061481.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [24]:
# VITS 多损失训练曲线（模拟）：KL/GAN/mel 一起下降
steps = np.arange(100)
kl = 0.5*np.exp(-steps/25) + 0.05
gan = 0.3*np.exp(-steps/40) + 0.1
mel = 0.8*np.exp(-steps/30) + 0.2
plt.plot(steps, kl, label='KL 损失(先验)')
plt.plot(steps, gan, label='GAN 损失(判别器)')
plt.plot(steps, mel, label='mel 重建损失')
plt.legend(fontsize=8); plt.xlabel('训练 step'); plt.yscale('log')
plt.title('VITS 三损失：KL 快速收敛，GAN/mel 缓慢（一阶段联合训练）')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_42700\833922359.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [25]:
# 韵律可视化：F0 曲线（重音在 0.35s 处）
t = np.arange(100)/100*1.2
f0 = 180 + 40*np.exp(-((t-0.35)/0.12)**2) + 15*np.sin(2*np.pi*1.5*t)*np.exp(-2*t)
plt.plot(t, f0, lw=2)
plt.axvspan(0.3, 0.4, alpha=0.2, color='C1', label='重音“好”')
plt.xlabel('时间(s)'); plt.ylabel('F0 (Hz)')
plt.legend(fontsize=8); plt.title('F0 轮廓：重音抬升 + 句末下降（韵律三件套之一）')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_42700\1975492204.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [26]:
# 说话人条件注入：embedding 广播到每一帧（加性）
emb = np.array([1.2, -0.8, 0.5])[:, None]
frame = np.random.RandomState(7).randn(3, 6)
cond = frame + emb
plt.figure(figsize=(7.5, 3.0))
plt.subplot(1, 2, 1)
plt.imshow(frame, aspect='auto', cmap='RdBu', vmin=-2, vmax=2)
plt.title('无条件的帧特征', fontsize=9)
plt.subplot(1, 2, 2)
plt.imshow(cond, aspect='auto', cmap='RdBu', vmin=-2, vmax=2)
plt.title('+说话人 embedding（整体偏移=音色）', fontsize=9)
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_42700\668927708.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 2️⃣2️⃣ MAS：VITS 的隐对齐目标

VITS 用**单调对齐搜索**找隐式时长（训练阶段）：

$$ \hat{A} = \arg\max_A \sum_{t=1}^{T} \log\, \mathcal N\big(x_t; \mu_{A(t)}, \sigma_{A(t)}\big) $$

约束：$A(t)$ 单调不回头、每个音素覆盖连续帧。搜索 = 在“帧×音素”网格上 DP（O(T·U)），与 DTW 同族。

找到 $\hat A$ 后用它监督时长预测器——**推理时不再需要对齐器**（与 FastSpeech 的外部 MFA 对齐器不同，MAS 全程自监督）。

In [27]:
# GST 风格 token 权重：参考音频分解为风格组合
styles = ['中性', '开心', '悲伤', '惊讶']
w = np.array([0.55, 0.18, 0.15, 0.12])
plt.bar(styles, w, color='C4')
plt.ylabel('风格 token 权重')
plt.title('GST：参考语音 → 风格向量加权（可插拔、可解耦）')
plt.tight_layout(); plt.show()
print('✓ 风格 token 可像条件一样拼接进解码器 → 控制情感/口音')

✓ 风格 token 可像条件一样拼接进解码器 → 控制情感/口音


C:\Users\24260\AppData\Local\Temp\ipykernel_42700\581448208.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [28]:
# 多频带声码器示意：全带波形 = 子带之和
t = np.arange(200)/100
s0 = np.sin(2*np.pi*3*t); s1 = 0.5*np.sin(2*np.pi*10*t); s2 = 0.25*np.sin(2*np.pi*25*t)
plt.figure(figsize=(7.5, 4.2))
plt.subplot(4, 1, 1); plt.plot(t, s0+s1+s2); plt.title('全带波形', fontsize=9)
for k, (s, nm) in enumerate([(s0, '子带1（低频/基频）'), (s1, '子带2（中频/共振峰）'),
                             (s2, '子带3（高频/清音）')], 2):
    plt.subplot(4, 1, k); plt.plot(t, s); plt.title(nm, fontsize=9)
plt.tight_layout(); plt.show()
print('✓ HiFi-GAN 多尺度判别器就同时看不同带宽/不同子带的真假')

✓ HiFi-GAN 多尺度判别器就同时看不同带宽/不同子带的真假


C:\Users\24260\AppData\Local\Temp\ipykernel_42700\1796515396.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [29]:
# MOS 评分分布：上线质量的“闸门”统计
rng = np.random.RandomState(8)
scores = np.clip(rng.normal(4.2, 0.5, 500), 1, 5)
plt.hist(scores, bins=20, density=True, alpha=0.6)
plt.axvline(4.0, color='r', ls='--', label='上线阈值 4.0')
plt.xlabel('MOS'); plt.ylabel('密度'); plt.legend(fontsize=8)
plt.title('MOS 分布：看均值还要看低分尾巴（长尾用户感知）')
plt.tight_layout(); plt.show()
print('低分(<3.0) 占比 = %.1f%%（长尾质量监控指标）' % (100*np.mean(scores < 3)))

低分(<3.0) 占比 = 1.2%（长尾质量监控指标）


C:\Users\24260\AppData\Local\Temp\ipykernel_42700\1643960280.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [30]:
# 数字示例：文本“你好”→ 波形 的每步形状与中间量
text = '你好'
phones = ['n', 'i', 'h', 'a', 'o']
dur = np.array([2, 4, 3, 5, 4])
T = int(dur.sum())
print('① 文本 = %s（%d 字）' % (text, len(text)))
print('② 前端 → 音素序列 =', phones, '（%d 个，含声母韵母）' % len(phones))
print('③ 时长预测（帧）=', dur.tolist(), '→ mel 总帧数 T = %d' % T)
print('④ 编码器输出形状 (T=%d, d=64) → mel 形状 (T=%d, 80)' % (T, T))
print('⑤ 声码器 → 波形 %d 采样 = %.2f s（@16kHz, 10ms/帧）' % (T*160, T*0.01))
print('✓ 每个中间量都可打印验证——面试“走一遍流程”题满分答案')

① 文本 = 你好（2 字）
② 前端 → 音素序列 = ['n', 'i', 'h', 'a', 'o'] （5 个，含声母韵母）
③ 时长预测（帧）= [2, 4, 3, 5, 4] → mel 总帧数 T = 18
④ 编码器输出形状 (T=18, d=64) → mel 形状 (T=18, 80)
⑤ 声码器 → 波形 2880 采样 = 0.18 s（@16kHz, 10ms/帧）
✓ 每个中间量都可打印验证——面试“走一遍流程”题满分答案


## 🎤 TTS 三面追问（8 连）

Q「GL 为什么迭代能恢复相位？」→ 幅度谱→时域→再取幅度谱投影，逐步逼近可行集；
Q「多损失怎么平衡？」→ 权重扫描 + 训练动态调整（mel 先稳、KL 后降）；
Q「MAS 与 MFA 区别？」→ MAS 无外部工具自监督；MFA 用外部标注器对齐；
Q「时长条上静音为什么重要？」→ 句间停顿=自然感的关键（短停+长停节奏）；
Q「子带波形有什么用？」→ 并行生成 + 多尺度判别器（GAN 更稳）；
Q「F0 曲线怎么控制？」→ 前端给重音/语调标签 → 韵律模型出 F0 → 声码器条件；
Q「风格 token 怎么训练？」→ 参考编码器 + 解耦约束（或软分类损失）；
Q「MOS 与业务指标怎么挂钩？」→ MOS 4.0+ 才敢放量；还要盯合成失败率/延迟分位。

In [31]:
# TTS 训练/推理双路流程框图（文本 → 音素 → 声学 → 波形）
fig, ax = plt.subplots(figsize=(9.5, 3.6))
st = [('文本\n(训练有标注)', '音素序列\n(字典映射)', 'Encoder\n+时长/对齐', 'mel 预测\nDecoder', '声码器\n(HiFi-GAN)', '波形')]
x0, w, gap, y, h = 0.02, 0.16, 0.006, 0.55, 0.32
for i, s in enumerate(st[0]):
    x = x0 + i*(w+gap)
    fc = '#ffe9d9' if i in (0, 1) else ('#dbe9ff' if i in (2, 3) else '#e2efda')
    ax.add_patch(plt.Rectangle((x, y), w, h, fc=fc, ec='#1f4e79', lw=1.2))
    ax.text(x+w/2, y+h/2, s, ha='center', va='center', fontsize=8)
    if i < 5:
        ax.annotate('', xy=(x+w+gap-0.001, y+h/2), xytext=(x+w+0.001, y+h/2),
                    arrowprops=dict(arrowstyle='->', lw=1.2, color='#1f4e79'))
ax.text(0.5, 0.12, '训练：mel 与波形为真值 → 损失(重建/时长/对抗)；推理：全部只前向，无需真值',
        ha='center', fontsize=9, color='#555')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('TTS 全景：文本侧(左) 与 声学侧(右)，声码器把 mel 变波形', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ VITS 把声码器并入训练(端到端)；Tacotron2 则 mel 与波形两阶段')

✓ VITS 把声码器并入训练(端到端)；Tacotron2 则 mel 与波形两阶段


C:\Users\24260\AppData\Local\Temp\ipykernel_42700\3390506424.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [32]:
# 单调对齐搜索（MAS）示意：文本 token 与 mel 帧的单调映射
fig, ax = plt.subplots(figsize=(7.8, 3.2))
mel = 8; tok = 4
for m in range(mel):
    for k in range(tok):
        ax.plot(m, k, 'o', color='#cfe3f5', ms=7)
path_m = np.linspace(0.4, mel-1.6, tok)
for k in range(tok):
    m0 = int(round(path_m[k]))
    ax.plot(m0, k, 'o', color='#b45309', ms=11)
    if k > 0:
        m1 = int(round(path_m[k-1]))
        ax.plot([m1, m0], [k-1, k], color='#b45309', lw=1.6)
ax.set_xlim(-0.6, mel); ax.set_ylim(-0.6, tok)
ax.set_xticks(range(mel)); ax.set_xticklabels(['mel t=%d' % i for i in range(mel)], rotation=45, fontsize=7)
ax.set_yticks(range(tok)); ax.set_yticklabels(['音素%d' % i for i in range(tok)], fontsize=8)
ax.set_title('MAS：每个音素一段连续帧，边界可跳不可回头（单调+覆盖）', fontsize=10)
plt.tight_layout(); plt.show()
print('✓ 对齐学习三选一：注意力(soft)、隐式(CTC式)、显式(MAS/Viterbi)')

✓ 对齐学习三选一：注意力(soft)、隐式(CTC式)、显式(MAS/Viterbi)


C:\Users\24260\AppData\Local\Temp\ipykernel_42700\1581826182.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🧮 TTS 核心公式（默写版）

**Tacotron2 损失**：$\mathcal{L}_{rec}=\|\hat m - m\|_2^2$（mel 重建）+ 逐帧停止概率交叉熵 $\mathcal{L}_{stop}$。

**VITS（VAE+GAN）总损失**：
$\mathcal{L}=\mathbb{E}_q[-\log p(x|z)] - D_{KL}(q(z\,|\,x)\parallel p(z\,|\,\text{text})) + \mathcal{L}_{adv}$。

**时长预测（VITS/非自回归）**：$\hat d_i=\sum_t A_{i,t}$（MAS 对齐矩阵逐音素求和）；训练用 $\mathcal{L}_{dur}=MSE(\hat d, d_{\text{true}})$，推理用预测时长展开帧。

**HiFi-GAN 对抗损失**：生成器 $\min\,\mathbb{E}[\|D(g(z))-1\|_2^2]$（多尺度/多周期判别器），加 mel 谱损失 $\|\text{STFT}(g)-\text{STFT}(x)\|_1$。

**Griffin-Lim 迭代**：$x^{(k)}=\text{ISTFT}\big(S\,e^{j\angle \text{STFT}(x^{(k-1)})}\big)$。

> 记忆轴：重建损失（mell/波形）+ 对齐机制（时长/注意力）+ 生成模型（VAE/GAN）= TTS 三大件。

In [33]:
# 时长展开数字示例：音素 + 帧数 → 帧级对齐
ph = np.array(['b', 'a', 'k'])
dur = np.array([3, 5, 2])
frames = np.repeat(np.arange(len(ph)), dur)
print('音素 =', list(ph), '| 时长 =', dur.tolist(), '| 帧级对齐 =', frames.tolist())
assert frames.tolist() == [0, 0, 0, 1, 1, 1, 1, 1, 2, 2]
print('✓ 3 音素展开成 10 帧；VITS 的时长预测模块就是在学这种展开')

音素 = [np.str_('b'), np.str_('a'), np.str_('k')] | 时长 = [3, 5, 2] | 帧级对齐 = [0, 0, 0, 1, 1, 1, 1, 1, 2, 2]
✓ 3 音素展开成 10 帧；VITS 的时长预测模块就是在学这种展开


In [34]:
# Griffin-Lim 数字示例：只有幅度谱 → 迭代恢复相位 → 误差下降
fs = 16000
t = np.arange(int(0.1*fs))/fs
x0 = np.sin(2*np.pi*440*t) + 0.4*np.sin(2*np.pi*1100*t)
win = 0.5*(1 - np.cos(2*np.pi*np.arange(400)/400)); hop = 160
S0 = stft(x0, win, hop)
mag = np.abs(S0)
X = mag*np.exp(1j*np.random.RandomState(7).rand(*mag.shape)*2*np.pi)
errs = []
for _ in range(12):
    xr = istft(X, win, hop, len(x0))
    Xs = stft(xr, win, hop)
    errs.append(np.linalg.norm(np.abs(Xs) - mag)/np.linalg.norm(mag))
    X = mag*np.exp(1j*np.angle(Xs))
plt.figure(figsize=(7.5, 2.8))
plt.plot(range(1, 13), errs, 'o-', lw=1.2, color='#1f4e79')
plt.xlabel('迭代'); plt.ylabel('幅度谱相对误差')
plt.title('Griffin-Lim：交替投影，误差单调下降（相位估计逐步逼近）', fontsize=10)
plt.tight_layout(); plt.show()
print('误差：迭代1 = %.4f → 迭代12 = %.4f' % (errs[0], errs[-1]))
assert errs[-1] < errs[0] * 0.6
print('✓ GL 收敛快但相位不可恢复原样 → 音质天花板低 → 学习式声码器取代')

误差：迭代1 = 0.5385 → 迭代12 = 0.2278
✓ GL 收敛快但相位不可恢复原样 → 音质天花板低 → 学习式声码器取代


C:\Users\24260\AppData\Local\Temp\ipykernel_42700\2437260441.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 📜 TTS 训练伪代码（40 行级）

```
for epoch:
  for (text, audio) in batch:
    phones = text2phone(text)                    # 文本 → 音素
    mel, dur_true = audio2mel(audio)             # 波形 → mel + 时长(可选标注)
    # Tacotron2：
    mel_pred, stop = decoder(encoder(phones))    # 自回归逐帧
    loss = L2(mel_pred, mel) + BCE(stop, eos_mask)
    # VITS：
    z = posterior_encoder(mel); d = duration_pred(align)   # 变分 + 时长
    loss = rec + KL(z) + dur_MSE + GAN_loss
    backward(); step()
```

**推理（VITS 一行）**：`waveform = vocoder(decoder(expand(phones, dur_pred(phones))))`。

> 面试加餐：为什么 VITS 能端到端？因为声码器也被纳入联合训练（GAN 判别器）。

## 🎤 TTS 追问（8 连·第二轮）

Q「Tacotron2 自回归慢，怎么办？」→ 非自回归(VITS/Flow-TTS) + 时长预测并行生成 mel；
Q「MAS 和 Attention 对齐区别？」→ Attention 软、可回退、易跳字；MAS 单调硬约束、更稳；
Q「时长模型怎么训练？」→ 用训练时对齐矩阵逐音素求和当真值 → 回归任务；
Q「HiFi-GAN 为什么多判别器？」→ 多尺度+多周期覆盖不同时域/频域模式 → 防模式坍塌；
Q「mel 能逆回波形吗？」→ 信息有损（相位/高频细节丢失）→ 声码器是从“统计分布”里补细节；
Q「VITS 的 V 指什么？」→ VAE：mel 编码到潜变量 z，再解码 → 生成多样性+端到端；
Q「怎么评测音质？」→ MOS 主观 + 客观(MCD/STOI/PESQ) 交叉验证；
Q「流式 TTS 延迟在哪？」→ 文本侧：首 token 前向；波形侧：chunk 级声码器 + 流水线并行。

## ✅ 自测（10 题）

1. TTS 两大流派？（自回归 Tacotron2 vs 非自回归 VITS/Flow）
2. 对齐为什么要单调？（语音时序物理约束：不能倒着读）
3. 时长预测输入输出？（音素 → 每音素帧数）
4. GL 缺什么？（相位）
5. HiFi-GAN 的 G 与 D 各自目标？（G 骗 D + mel 一致；D 区分真假）
6. VITS 的 KL 项在哪？（后验 q(z|x) 与先验 p(z|text)）
7. 为什么 MOS 不能只看一个评分？（主观方差、男女/口音覆盖）
8. 声码器输入输出？（mel → 波形）
9. 合成与识别共享什么？（声学特征 mel、对齐思想）
10. 端到端 TTS 的好处？（联合优化、无中间误差累积）

In [35]:
# 声码器权衡对比：音质 / 速度 / 工程成熟度
vocs = ['Griffin-Lim', 'WaveNet\n自回归', 'HiFi-GAN\n(GAN)', 'HiFi-GAN\n+mel 损失']
vals = {'音质': [60, 90, 95, 96], '速度': [99, 25, 70, 68], '成熟度': [95, 80, 85, 88]}
x = np.arange(4)
plt.figure(figsize=(7.8, 3.2))
for j, (k, v) in enumerate(vals.items()):
    plt.bar(x + (j-1)*0.27, v, 0.27, label=k, color=['#1f4e79', '#376027', '#b45309'][j])
plt.xticks(x, vocs, fontsize=8); plt.ylabel('相对评分 (0-100)')
plt.legend(fontsize=8); plt.title('声码器三角权衡：学习式全面胜出，但需训练与 GPU')
plt.tight_layout(); plt.show()
print('✓ 面试话术：GL 兜底/无训练；WaveNet 音质好但慢；HiFi-GAN 是当前工业标准')

✓ 面试话术：GL 兜底/无训练；WaveNet 音质好但慢；HiFi-GAN 是当前工业标准


C:\Users\24260\AppData\Local\Temp\ipykernel_42700\1534239474.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [36]:
# 时长预测数字示例：由对齐矩阵 A 求每音素时长，再反推 mel 帧归属
A = np.array([[1, 0, 0, 0, 0],     # 音素0覆盖 mel 帧 0
              [0, 1, 1, 1, 0],     # 音素1覆盖 1,2,3
              [0, 0, 0, 0, 1]])    # 音素2覆盖 4
d = A.sum(axis=1)
print('时长预测 =', d.tolist(), '（= 对齐矩阵逐行求和）')
expand = np.repeat(np.arange(len(d)), d)
print('展开 =', expand.tolist())
assert d.tolist() == [1, 3, 1]
print('✓ 训练：A 来自 MAS/Attention → 真值时长；推理：时长网络直接预测 → 并行展开')

时长预测 = [1, 3, 1] （= 对齐矩阵逐行求和）
展开 = [0, 1, 1, 1, 2]
✓ 训练：A 来自 MAS/Attention → 真值时长；推理：时长网络直接预测 → 并行展开


## 🍵 白话开篇：TTS 到底分几步？

语音识别是把“声音 → 文字”，语音合成反过来：**“文字 → 声音”**。
一整句话怎么变成能播的波形？主流做法分**两段接力**：

```
文字 → [① 前端：分词/注音/韵律] → [② 声学模型：文字→mel 谱]
      → [③ 声码器：mel 谱→波形] → 播放
```

**比喻**：
- **声学模型** = 把“演讲稿”变成“五线谱”（哪些时刻什么音高/音色）；
- **声码器** = 把“五线谱”变成“真人演唱”（波形）；
- 前端 = 先标好断句重音（不然“下雨天留客天留我不留”会读错）。

**为什么中间产物选 mel 谱**：mel 谱 = “人耳最关心的频谱摘要”，维度低（26~80 维/帧）好生成，
而且声码器能高质量重建。**代价**：mel 丢了相位——波形要靠声码器“脑补”相位。

**一句话**：TTS = 文本→谱（学“怎么说”） + 谱→波（学“怎么响”），两部分独立优化。

## 🍵 白话讲透 Tacotron2 注意力：逐字念稿的“提词器”

Tacotron2 的 Decoder 每生成一帧 mel，都要回答：**“我该读稿子上哪个字了？”**
——这就是注意力做的事。

**画面**：
- Encoder 把每个字编码成一张“卡片”（h₁..h_N）；
- Decoder 拿“当前写到哪”的隐状态，给每张卡片打分 → softmax → 加权取内容；
- 注意力矩阵热图：横轴字、纵轴帧——**理想是条从左下到右上的斜线**（逐字推进、不跳字）。

**为什么要“位置敏感”**：普通注意力可能“回头重读”（卡住循环）或“跳字”（漏读）。
Tacotron2 在打分里加**上一帧注意力位置**的偏置：
注意力只能**单调向前挪**一点 → 大幅减少漏字/重读。

**单调性为什么重要**：人朗读时字序不回头——违反单调的对齐听起来就是“卡碟/串词”。

**停止条件**：Decoder 还输出一个“stop 概率”——它判断“读完了吗”→
读完就停，否则会无限生成（合成的经典翻车：一直嗡嗡）。

## 🍵 白话讲透声码器：怎么把谱“唱”成波形？

声码器（vocoder）吃 mel 谱，吐出波形。三代演进：

1. **传统参数式**（Griffin-Lim / STRAIGHT / WORLD）：
   - Griffin-Lim：只有幅度谱没有相位 → 先瞎猜一个相位，来回“谱域↔时域”投影，
     每次投影让“相位与幅度更自洽”，几十轮后误差下降（本书 r9 已验证单调下降）；
   - 特点：不用训练、纯迭代，但音质一般、速度慢。
2. **神经声码器**（WaveNet / WaveGlow / HiFi-GAN）：
   - 让网络**直接学“谱 → 波形”**的映射（自回归或生成对抗）；
   - HiFi-GAN：生成器合成波形、判别器判真假 → 逼着波形像真人（目前音质标杆之一）。
3. **端到端免声码器**（VITS）：
   - 直接把波形当输出训练（flow + VAE），不再显式重建 mel → 一条路到底。

**为什么声码器是“最后 10%”**：前面模型决定“说什么、什么调”，声码器决定“像不像真人”
（呼吸感、颗粒感、自然度）——MOS 分数的大头往往差在这里。

## 🍵 白话讲透时长模型：谁决定每个字念多久？

一句话“你好”= 2 个字，但 mel 有 60 帧——**谁来决定“你”占 30 帧、“好”占 30 帧？**

**训练时（监督信号来自对齐）**：
- Tacotron2：注意力矩阵自动学出“字→帧”的对齐 → 时长 = 对齐矩阵逐行求和（本书 r9 示例）；
- FastSpeech2 / VITS：用外部对齐工具（如 MFA）或 MAS（单调对齐搜索）抽时长当标签。

**推理时（要并行、不能等逐帧念）**：
- 训练一个**时长预测器**：输入字向量 → 输出“每个字多少帧”；
- 把文本按预测时长**展开**成逐字序列 → 一次并行生成整条 mel（比自回归快 10 倍+）。

**为什么时长难学**：语速因人因语境而异；时长错一点，听感就是“结巴”或“机关枪”。

**一句话**：时长 = 文本到语音的“节奏编排”——自回归靠注意力隐式学，并行模型靠显式预测器。

## 🔍 公式拆解 ①：Tacotron2 的损失

$$\mathcal{L}=\underbrace{\|m-\hat m\|_2^2}_{\text{mel 谱 MSE}}\ +\ \underbrace{\text{BCE}(s,\hat s)}_{\text{停止}}\ +\ \underbrace{\|w-\hat w\|_2^2}_{\text{辅助门控}}(+\text{后置网络 MSE})$$

**逐符号解释**：
- $m$：真实 mel 谱（训练数据里抽的）；$\hat m$：模型生成的 mel 谱；
- $\|\cdot\|_2^2$：逐元素平方误差求和——**逐帧逐带**地比“声音参数像不像”；
- $s$：停止标志（1=读完了）；$\hat s$：模型预测的停止概率；BCE = 二分类交叉熵；
- $w$：线性谱（额外监督信号）——让 Decoder 顺便学“细一点”的谱；
- 后置网络（PostNet）：对 mel 再精修，防“糊”（欠采样平滑）。

**为什么用 MSE 不用别的**：mel 是连续值（能量），回归任务默认 L2；
MSE 对“平均值漂移”敏感、对孤立异常值不太敏感——合成场景够用。

**直觉**：损失 = “生成的谱 vs 真实谱”的逐点差距 + “该停没停”的判定错误。
两项一个管音质、一个管长度——训练里两者都要。

## 🔍 公式拆解 ②：VITS 的两个关键模块

**VAE 部分（先验/后验）**：
$$\mathcal{L}_{VAE} = -\underbrace{\mathbb{E}_{q(z\mid m)}[\log p(m\mid z)]}_{\text{重建：波形/谱要像}} + \underbrace{D_{KL}\big(q(z\mid m)\,\|\,p(z\mid t)\big)}_{\text{正则：隐变量别跑太远}}$$

**逐符号解释**：
- $z$：隐变量（“这段语音的说话方式”的浓缩编码）；$m$：mel/波形；$t$：文本；
- $q(z|m)$：后验（编码器：从声音猜隐变量）；$p(z|t)$：先验（从文本猜隐变量）；
- 第一项：从 z 重建 m 要准（生成得像）；第二项：z 别偏离文本预测太多（声音得对得上字）。

**Flow 部分（可逆变换）**：
$$\log p_Z(z)=\log p_U(f^{-1}(z)) + \log\left|\det\frac{\partial f^{-1}}{\partial z}\right|$$

**逐符号解释**：
- $f$：可逆变换（一堆可逆层把简单分布 U 变成复杂分布 Z）；
- $f^{-1}$：逆变换（推理/采样时从 Z 变回 U 用）；
- $\log|\det J|$：体积变化修正项——变换拉伸了多少，概率密度就要缩回多少；
- 好处：**精确可计算**的密度（不像 GAN 只能对抗），可直接最大似然训练。

**一句话**：VITS = VAE 管“像不像+对得上”，Flow 管“概率算得准”，两者拼接成端到端。

## 🔍 公式拆解 ③：Griffin-Lim 为什么能收敛？

$$\hat x^{(k+1)}=\operatorname{ISTFT}\Big(M\cdot e^{j\angle\operatorname{STFT}(\hat x^{(k)})}\Big)$$

**逐符号解释**：
- $M$：给定的幅度谱（目标）；$\hat x^{(k)}$：第 k 轮估计的波形；
- $\angle\operatorname{STFT}(\hat x^{(k)})$：当前波形谱的**相位**（先瞎猜/上轮结果）；
- 操作：把“上轮相位的相位 + 目标幅度”反变换回波形；
- 每轮 = **在“有目标幅度”和“是真实信号”两个集合间来回投影**。

**为什么迭代会变好（交替投影直觉）**：
设 A = 幅度满足目标的信号集合，B = 能由某个波形 STFT 出来的信号集合（一致性）。
每步投影都让波形**更接近 B∩A**——虽然投影可能“踩一脚、退半步”，
但总体误差单调下降（本书 r9 已验证：12 轮误差降 58%）。

**局限**：只能找**与目标幅度谱一致**的波形，但“真实相位”可能不在 A∩B 内 → 音质天花板低；
神经声码器正是为了跳过这个限制。

In [37]:
# 声码器本质演示：只有幅度谱（丢相位），重建误差多大？GL 能救回多少？
fs = 16000
t = np.arange(int(0.1*fs))/fs
x0 = np.sin(2*np.pi*440*t) + 0.5*np.sin(2*np.pi*990*t) + 0.2*np.sin(2*np.pi*1490*t)
win = np.hamming(400)                                # 端点非零 → ISTFT 归一化不除零
hop = 160
M = np.abs(stft(x0, win, hop))                       # 只要幅度谱
rng = np.random.RandomState(3)
Xr = M*np.exp(1j*rng.rand(*M.shape)*2*np.pi)         # 相位全随机
e0 = np.linalg.norm(istft(Xr, win, hop, len(x0)) - x0)/np.linalg.norm(x0)
errs = []
X = Xr
for _ in range(25):
    X = M*np.exp(1j*np.angle(stft(istft(X, win, hop, len(x0)), win, hop)))
    errs.append(np.linalg.norm(istft(X, win, hop, len(x0)) - x0)/np.linalg.norm(x0))
print('丢相位直接重建误差 = %.3f（相位随机 → 波形乱成一团）' % e0)
print('GL 迭代后误差 = %.3f → %.3f（25 轮下降 %.0f%%）' % (errs[0], errs[-1], 100*(1-errs[-1]/errs[0])))
assert errs[-1] < errs[0]
print('✓ 结论：相位不可丢；丢了解不回来，只能靠 GL 迭代或神经声码器“补脑”')

丢相位直接重建误差 = 1.844（相位随机 → 波形乱成一团）
GL 迭代后误差 = 1.603 → 1.114（25 轮下降 31%）
✓ 结论：相位不可丢；丢了解不回来，只能靠 GL 迭代或神经声码器“补脑”


In [38]:
# 单调注意力演示：正常斜线 vs 乱序（漏字/重读）热图对比
n_txt, n_fr = 6, 40
diag = np.zeros((n_fr, n_txt))
for i in range(n_fr):
    j = min(n_txt-1, i*n_txt//n_fr)
    diag[i, max(0, j-1):j+2] = 0.5
    diag[i, j] = 1.0
diag = diag/np.maximum(diag.sum(axis=1, keepdims=True), 1e-9)
scramble = diag[[0, 1, 2, 3, 2, 1, 2, 3, 4, 5, 3, 4, 5, 5, 4, 5]*2+[5]*9]  # 乱序/回头
fig, ax = plt.subplots(1, 2, figsize=(8.5, 3.0))
im1 = ax[0].imshow(diag.T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax[0].set_title('正常：单调斜线（逐字推进）', fontsize=9)
ax[0].set_xlabel('帧'); ax[0].set_ylabel('字')
im2 = ax[1].imshow(scramble.T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax[1].set_title('异常：回头/跳字（漏字重读）', fontsize=9)
ax[1].set_xlabel('帧'); ax[1].set_ylabel('字')
plt.tight_layout(); plt.show()
print('✓ 训练时正则/单调约束就是把这些“回头”热图压成左图——防串词防漏字')

✓ 训练时正则/单调约束就是把这些“回头”热图压成左图——防串词防漏字


C:\Users\24260\AppData\Local\Temp\ipykernel_42700\3226034766.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## ⚠️ 本章高频踩坑

1. **注意力崩溃**：AED/Tacotron 长句易“重读循环”（一直嗡嗡）——加单调约束/位置偏置；
2. **时长与语速**：并行模型时长预测错 5% 就“结巴”——时长预测器要单独训好；
3. **相位问题**：mel 丢相位 → 波形必须靠声码器；直接拿 GL 音频做训练标签会引入“金属味”；
4. **帧率转换**：声学模型帧移(如 12.5ms) 与声码器帧移(如 10ms) 不一致 → 长度不匹配，需重采样；
5. **stop token 训练不均**：短句 stop 多、长句 stop 少 → 样本不均衡，需加权；
6. **MOS 是主观的**：同一系统不同评测差 0.3+ 分——报告要注明评测池/参考系；
7. **多说话人**：说话人 embedding 维度太低会“音色平均化”——每人一个向量要留容量；
8. **文本前端**：数字/缩写/多音字不展开，合成必错（TTS 的 30% 错误来自文本规则）。

## 🧭 TTS 代际对比：Tacotron2 / FastSpeech2 / VITS / VALL-E

| 模型 | 结构 | 并行 | 声码器 | 特点 |
|---|---|---|---|---|
| Tacotron2 | AED+注意力 | ✗ 自回归 | 需（WaveGlow/HiFi-GAN） | 经典、质量好、慢 |
| FastSpeech2 | 非自回归+时长预测 | ✓ | 需 | 快、可控（时长/音高/能量可调） |
| VITS | VAE+Flow+GAN | ✓ | 免（直接波形） | 端到端、一阶段、质量高 |
| VALL-E | 神经编解码（AudioLM 系） | 部分 | 免 | 少样本克隆、零样本 TTS |

**选型逻辑**：
- 要**快+稳**（线上）→ FastSpeech2 系；
- 要**质量+端到端** → VITS 系；
- 要**音色克隆/少样本** → VALL-E/GPT-SoVITS 系（大模型路线）；
- 学习理解 → 先啃 Tacotron2（注意力、stop、postnet 概念最全）。

**趋势**：从“两段式+声码器”走向“大模型直接出波形/离散码”（Neural Codec + LLM），
但 mel 作为中间表示仍被广泛用于训练与调试。

## 🏁 本章学完清单

- [ ] 能画出 TTS 三段流水线（前端/声学/声码器）并解释各管什么；
- [ ] 能用“提词器”讲清注意力单调性为什么重要；
- [ ] 能写出 Tacotron2 损失公式并解释每个符号；
- [ ] 能解释 VITS 里 VAE 两项各管什么、Flow 的 log|det J| 为什么必须乘；
- [ ] 能解释 GL 交替投影为什么误差单调下降（r9 数字佐证）；
- [ ] 能对比三代声码器（GL/神经/端到端）取舍；
- [ ] 能解释时长预测器为什么让推理并行化；
- [ ] 能说出 3 个 TTS 工程坑（注意力崩溃/时长/MOS 评测）。

> 下一站：语音工程全景——VAD/回声消除/波束/声纹（下一本 notebook）。